<a href="https://colab.research.google.com/github/djqx7/iasnlp_2026/blob/main/llm/speach_text_input.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q --no-cache-dir \
  "transformers==4.52.3" \
  "tokenizers==0.21.4" \
  "accelerate>=1.2.0" \
  "huggingface_hub>=0.30.0" \
  "qwen-omni-utils" \
  soundfile librosa pandas scikit-learn tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 167.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 249.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 165.9 MB/s eta 0:00:00


In [5]:
from google.colab import files

uploaded = files.upload()

Saving test_dataset_final.csv to test_dataset_final.csv


In [3]:
  import transformers
import tokenizers
import huggingface_hub

print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)

from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
from qwen_omni_utils import process_mm_info

print("Qwen2.5 Omni imports successful")
import zipfile
import os

zip_path = "audio_files.zip"
audio_dir = "audio_files"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(".")

print("Files extracted.")

print("Checking audio folder:")
print(os.listdir(audio_dir)[:10])
print("Total audio files:", len(os.listdir(audio_dir)))

transformers: 4.52.3
tokenizers: 0.21.4
huggingface_hub: 0.36.2
Qwen2.5 Omni imports successful
Files extracted.
Checking audio folder:
['0_1_d1680.wav', '12_0_d66.wav', '0_0_d749.wav', '1_1_d2027.wav', '1_0_d706.wav', '2_0_d1891.wav', '1_0_d762.wav', '0_1_d1817.wav', '1_0_d2539.wav', '12_1_d1292.wav']
Total audio files: 404


In [6]:
import os
import re
import pandas as pd
import torch
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
from qwen_omni_utils import process_mm_info

file_path = "test_dataset_final.csv"
audio_dir = "audio_files"

df = pd.read_csv(file_path)

print("CSV preview:")
print(df.head())
print("Columns:", df.columns)

text_col = "transcript"
audio_col = "file_name"
label_col = "label"

df = df.dropna(subset=[text_col, audio_col, label_col]).reset_index(drop=True)

labels = sorted(df[label_col].astype(str).unique().tolist())

print("Audio column:", audio_col)
print("Text column:", text_col)
print("Label column:", label_col)
print("Labels:", labels)

missing = []

for f in df[audio_col].astype(str).tolist():
    p = os.path.join(audio_dir, f)
    if not os.path.exists(p):
        missing.append(f)

print("Missing audio files:", len(missing))

if len(missing) > 0:
    print("First missing files:", missing[:10])
    raise FileNotFoundError("Some audio files from CSV are not present in audio_files folder.")

print("\nChecking first 3 audio-text pairs:")
for i in range(min(3, len(df))):
    audio_path = os.path.join(audio_dir, str(df.loc[i, audio_col]))
    transcript = str(df.loc[i, text_col])
    label = str(df.loc[i, label_col])

    print("\nSample", i)
    print("Audio path:", audio_path)
    print("Audio exists:", os.path.exists(audio_path))
    print("Audio size bytes:", os.path.getsize(audio_path))
    print("Transcript:", transcript)
    print("True label:", label)

model_name = "Qwen/Qwen2.5-Omni-3B"

print("\nLoading model:", model_name)

model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

processor = Qwen2_5OmniProcessor.from_pretrained(model_name)

model.eval()

print("Model loaded successfully.")

def make_prompt(transcript):
    prompt = f"""You are a sentence type classifier.

You will receive both the speech audio and its transcript.

Classify the sentence into exactly one of these labels:

declarative: a normal statement or information-giving sentence.
interrogative: a question or sentence asking for information.
imperative: a command, request, advice, instruction, or order.
exclamatory: a sentence showing strong emotion, surprise, excitement, or emphasis.

Use both the audio and the transcript. The transcript gives the words, and the audio may give tone, emphasis, and intonation.

Examples:
Sentence: I am going to school today.
Label: declarative

Sentence: What are you doing here?
Label: interrogative

Sentence: Please close the door.
Label: imperative

Sentence: What a beautiful day!
Label: exclamatory

Sentence: Give me scotch, please.
Label: imperative

Sentence: I heard you and James are engaged at last.
Label: declarative

Sentence: Hi, what brings you to my office today?
Label: interrogative

Now classify this sentence.

Transcript:
{transcript}

Return only one label from:
declarative, exclamatory, imperative, interrogative

Answer with only the label.

Label:"""

    return prompt

def clean_pred(ans):
    raw = ans
    ans = ans.strip().lower()

    ans = ans.replace("answer:", " ")
    ans = ans.replace("label:", " ")

    ans = re.sub(r"[^a-zA-Z]", " ", ans)
    ans = " ".join(ans.split())

    words = ans.split()

    for lab in labels:
        if ans == lab.lower():
            return lab

    for w in words:
        for lab in labels:
            if w == lab.lower():
                return lab

    for lab in labels:
        if lab.lower() in ans:
            return lab

    print("Could not clean prediction. Raw answer was:", raw)
    return ans

def print_input_debug(i, audio_path, transcript, conversation, text, audios, images, videos, inputs):
    print("\n" + "=" * 80)
    print("DEBUG SAMPLE:", i)
    print("Audio path:", audio_path)
    print("Audio exists:", os.path.exists(audio_path))
    print("Audio size bytes:", os.path.getsize(audio_path))
    print("Transcript:", transcript)

    print("\nConversation content types:")
    for msg in conversation:
        print("Role:", msg["role"])
        for item in msg["content"]:
            print("  type:", item["type"])

    print("\nChat template first 600 characters:")
    print(text[:600])

    print("\nMultimodal objects after process_mm_info:")
    print("Number of audios:", 0 if audios is None else len(audios))
    print("Number of images:", 0 if images is None else len(images))
    print("Number of videos:", 0 if videos is None else len(videos))

    if audios is not None and len(audios) > 0:
        a0 = audios[0]
        print("First audio object type:", type(a0))

        if hasattr(a0, "shape"):
            print("First audio shape:", a0.shape)

        if hasattr(a0, "dtype"):
            print("First audio dtype:", a0.dtype)

    print("\nProcessor input keys:")
    print(list(inputs.keys()))

    print("\nProcessor input tensor details:")
    for k, v in inputs.items():
        if hasattr(v, "shape"):
            print(k, "shape:", tuple(v.shape), "dtype:", v.dtype, "device before move:", v.device)
        else:
            print(k, "type:", type(v))

    if "input_ids" in inputs:
        print("Text input token length:", inputs["input_ids"].shape[1])

    print("=" * 80 + "\n")

def predict(audio_path, transcript, i=None, debug=False):
    prompt = make_prompt(transcript)

    conversation = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": "You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of perceiving auditory inputs and generating text responses."
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "audio",
                    "audio": audio_path
                },
                {
                    "type": "text",
                    "text": prompt
                }
            ],
        }
    ]

    text = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=False
    )

    audios, images, videos = process_mm_info(
        conversation,
        use_audio_in_video=False
    )

    inputs = processor(
        text=text,
        audio=audios,
        images=images,
        videos=videos,
        return_tensors="pt",
        padding=True,
        use_audio_in_video=False
    )

    if debug:
        print_input_debug(i, audio_path, transcript, conversation, text, audios, images, videos, inputs)

    input_len = inputs["input_ids"].shape[1]

    inputs = inputs.to(model.device)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            use_audio_in_video=False,
            return_audio=False,
            max_new_tokens=10,
            do_sample=False
        )

    if isinstance(out_ids, tuple):
        out_ids = out_ids[0]

    full_text = processor.batch_decode(
        out_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    new_ids = out_ids[:, input_len:]

    new_text = processor.batch_decode(
        new_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    if debug:
        print("\nRaw decoded FULL text:")
        print(full_text)

        print("\nRaw decoded NEW text only:")
        print(new_text)

    if len(new_text.strip()) > 0:
        ans = new_text
    else:
        ans = full_text.split("Label:")[-1]

    pred = clean_pred(ans)

    if debug:
        print("Cleaned prediction:", pred)
        print("-" * 80)

    return pred

print("\nRunning debug predictions on first 5 samples only:")

debug_preds = []

for i in range(min(5, len(df))):
    audio_path = os.path.join(audio_dir, str(df.loc[i, audio_col]))
    transcript = str(df.loc[i, text_col])
    true_label = str(df.loc[i, label_col])

    pred = predict(audio_path, transcript, i=i, debug=True)

    debug_preds.append(pred)

    print("Sample:", i)
    print("True label:", true_label)
    print("Predicted label:", pred)

print("\nDebug first 5 predictions:", debug_preds)

print("\nNow running on full dataset...")

preds = []

for i, row in tqdm(df.iterrows(), total=len(df)):
    audio_path = os.path.join(audio_dir, str(row[audio_col]))
    transcript = str(row[text_col])

    pred = predict(audio_path, transcript, i=i, debug=False)
    preds.append(pred)

df["predicted_type"] = preds

print("\nPrediction counts:")
print(df["predicted_type"].value_counts())

print("\nFirst 20 predictions:")
print(df[[audio_col, text_col, label_col, "predicted_type"]].head(20))

y_true = df[label_col].astype(str).tolist()
y_pred = df["predicted_type"].astype(str).tolist()

acc = accuracy_score(y_true, y_pred)

p, r, f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

print("Accuracy:", acc)
print("Precision:", p)
print("Recall:", r)
print("F1 Score:", f1)

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, zero_division=0))

df.to_csv("qwen25_omni_speech_text_predictions_fixed.csv", index=False)

print("Saved predictions to qwen25_omni_speech_text_predictions_fixed.csv")

from google.colab import files
files.download("qwen25_omni_speech_text_predictions_fixed.csv")

CSV preview:
      file_name                                         transcript  \
0   0_0_d45.wav     Hello, Jane. I'm really glad to meet you here.   
1   0_0_d47.wav         I heard you and James are engaged at last.   
2  0_0_d215.wav                            Give me scotch, please.   
3   0_0_d49.wav            Hi, what brings you to my office today?   
4  0_0_d159.wav  Please help yourself at your dishes. I hope yo...   

           label  
0    declarative  
1    declarative  
2     imperative  
3  interrogative  
4     imperative  
Columns: Index(['file_name', 'transcript', 'label'], dtype='object')
Audio column: file_name
Text column: transcript
Label column: label
Labels: ['declarative', 'exclamatory', 'imperative', 'interrogative']
Missing audio files: 0

Checking first 3 audio-text pairs:

Sample 0
Audio path: audio_files/0_0_d45.wav
Audio exists: True
Audio size bytes: 242594
Transcript: Hello, Jane. I'm really glad to meet you here.
True label: declarative

Sample 1
Aud

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section'}


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...


Raw decoded FULL text:
system
You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of perceiving auditory inputs and generating text responses.
user
You are a sentence type classifier.

You will receive both the speech audio and its transcript.

Classify the sentence into exactly one of these labels:

declarative: a normal statement or information-giving sentence.
interrogative: a question or sentence asking for information.
imperative: a command, request, advice, instruction, or order.
exclamatory: a sentence showing strong emotion, surprise, excitement, or emphasis.

Use both the audio and the transcript. The transcript gives the words, and the audio may give tone, emphasis, and intonation.

Examples:
Sentence: I am going to school today.
Label: declarative

Sentence: What are you doing here?
Label: interrogative

Sentence: Please close the door.
Label: imperative

Sentence: What a beautiful day!
Label: exclamatory

Sentence: Give me scotch, please.
Label

100%|██████████| 404/404 [14:57<00:00,  2.22s/it]


Prediction counts:
predicted_type
interrogative    135
declarative      112
exclamatory      106
imperative        51
Name: count, dtype: int64

First 20 predictions:
        file_name                                         transcript  \
0     0_0_d45.wav     Hello, Jane. I'm really glad to meet you here.   
1     0_0_d47.wav         I heard you and James are engaged at last.   
2    0_0_d215.wav                            Give me scotch, please.   
3     0_0_d49.wav            Hi, what brings you to my office today?   
4    0_0_d159.wav  Please help yourself at your dishes. I hope yo...   
5     0_0_d50.wav                                  Hello, Jack here.   
6    0_0_d403.wav            Excuse me. I wonder if you can help me.   
7     0_0_d63.wav            Umm, Jenny, are you having a good time?   
8    0_0_d681.wav  Excuse me. I bought this shirt yesterday, but ...   
9    0_0_d684.wav     Watch out! You're too close to the fire place.   
10   0_0_d704.wav                       

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
import os
import re
import pandas as pd
import torch

from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor

file_path = "test_dataset_final.csv"
audio_dir = "audio_files"

# ============================================================
# Load CSV
# ============================================================

df = pd.read_csv(file_path)

print("CSV preview:")
print(df.head())
print("Columns:", df.columns)

text_col = "transcript"
audio_col = "file_name"
label_col = "label"

df = df.dropna(subset=[audio_col, label_col]).reset_index(drop=True)

labels = sorted(df[label_col].astype(str).unique().tolist())

print("\nRunning Qwen2.5-Omni speech-only evaluation")
print("Audio column:", audio_col)
print("Label column:", label_col)
print("Labels:", labels)

# ============================================================
# Check audio files
# ============================================================

missing = []

for f in df[audio_col].astype(str).tolist():
    p = os.path.join(audio_dir, f)
    if not os.path.exists(p):
        missing.append(f)

print("Missing audio files:", len(missing))

if len(missing) > 0:
    print("First missing files:", missing[:10])
    raise FileNotFoundError("Some audio files are missing.")

print("\nChecking first 3 audio files:")

for i in range(min(3, len(df))):
    audio_path = os.path.join(audio_dir, str(df.loc[i, audio_col]))

    print("\nSample", i)
    print("Audio path:", audio_path)
    print("Audio exists:", os.path.exists(audio_path))
    print("Audio size bytes:", os.path.getsize(audio_path))
    print("True label:", str(df.loc[i, label_col]))

# ============================================================
# Load Qwen2.5-Omni model if not already loaded
# ============================================================

model_name = "Qwen/Qwen2.5-Omni-3B"

if "processor" not in globals() or "model" not in globals():
    print("\nLoading model:", model_name)

    if not torch.cuda.is_available():
        raise RuntimeError("Please enable GPU in Colab: Runtime -> Change runtime type -> GPU")

    processor = Qwen2_5OmniProcessor.from_pretrained(model_name)

    model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True
    ).eval()

    print("Model and processor loaded successfully.")
else:
    print("\nUsing already loaded model and processor.")

# ============================================================
# Prompt
# ============================================================

def make_qwen_speech_only_prompt():
    prompt = """You are a sentence type classifier.

Listen to the speech audio carefully.

Classify the spoken sentence into exactly one of these labels:

declarative: a normal statement or information-giving sentence.
interrogative: a question or sentence asking for information.
imperative: a command, request, advice, instruction, or order.
exclamatory: a sentence showing strong emotion, surprise, excitement, or emphasis.

Use only the speech audio. Do not use any transcript.

Return only one label from:
declarative, exclamatory, imperative, interrogative

Answer with only the label.

Label:"""

    return prompt

# ============================================================
# Prediction cleaner
# ============================================================

def clean_pred_qwen_speech_only(ans):
    raw = ans
    ans = ans.strip().lower()

    ans = ans.replace("answer:", " ")
    ans = ans.replace("label:", " ")

    ans = re.sub(r"[^a-zA-Z]", " ", ans)
    ans = " ".join(ans.split())

    words = ans.split()

    for lab in labels:
        if ans == lab.lower():
            return lab

    for w in words:
        for lab in labels:
            if w == lab.lower():
                return lab

    for lab in labels:
        if lab.lower() in ans:
            return lab

    print("Could not clean prediction. Raw answer was:", raw)
    return ans

# ============================================================
# Helper to find model input device
# ============================================================

def get_model_input_device():
    try:
        return model.device
    except Exception:
        for p in model.parameters():
            if p.device.type != "meta":
                return p.device

    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# Debug printer
# ============================================================

def print_qwen_speech_only_debug(i, audio_path, conversation, inputs):
    print("\n" + "=" * 80)
    print("DEBUG SAMPLE:", i)
    print("Audio path:", audio_path)
    print("Audio exists:", os.path.exists(audio_path))
    print("Audio size bytes:", os.path.getsize(audio_path))

    print("\nConversation content types:")
    for msg in conversation:
        print("Role:", msg["role"])
        for item in msg["content"]:
            print("  type:", item["type"])

    print("\nProcessor input keys:")
    print(list(inputs.keys()))

    print("\nProcessor input tensor details:")
    for k, v in inputs.items():
        if torch.is_tensor(v):
            print(k, "shape:", tuple(v.shape), "dtype:", v.dtype, "device before move:", v.device)
        else:
            print(k, "type:", type(v))

    if "input_ids" in inputs:
        print("Text input token length:", inputs["input_ids"].shape[-1])

    audio_keys = [
        k for k in inputs.keys()
        if "audio" in k.lower() or "feature" in k.lower()
    ]

    print("Audio-related keys:", audio_keys)
    print("=" * 80 + "\n")

# ============================================================
# Build Qwen inputs
# ============================================================

def build_qwen_speech_only_inputs(audio_path):
    prompt = make_qwen_speech_only_prompt()

    conversation = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": "You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of perceiving auditory and visual inputs, as well as generating text and speech."
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "audio",
                    "path": audio_path
                },
                {
                    "type": "text",
                    "text": prompt
                }
            ],
        }
    ]

    try:
        inputs = processor.apply_chat_template(
            conversation,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            padding=True,
            use_audio_in_video=False
        )
    except TypeError:
        inputs = processor.apply_chat_template(
            conversation,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt"
        )

    return conversation, inputs

# ============================================================
# Prediction function
# ============================================================

def predict_qwen_speech_only(audio_path, i=None, debug=False):
    conversation, inputs = build_qwen_speech_only_inputs(audio_path)

    if debug:
        print_qwen_speech_only_debug(i, audio_path, conversation, inputs)

    audio_keys = [
        k for k in inputs.keys()
        if "audio" in k.lower() or "feature" in k.lower()
    ]

    if len(audio_keys) == 0:
        raise ValueError(
            "Audio was not processed. Processor output has no audio or feature keys. "
            "Check that the audio message uses {'type': 'audio', 'path': audio_path}."
        )

    input_len = inputs["input_ids"].shape[-1]

    device = get_model_input_device()
    inputs = inputs.to(device)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            use_audio_in_video=False,
            return_audio=False,
            max_new_tokens=10,
            do_sample=False
        )

    if isinstance(out_ids, tuple):
        out_ids = out_ids[0]

    new_ids = out_ids[:, input_len:]

    new_text = processor.batch_decode(
        new_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0].strip()

    full_text = processor.batch_decode(
        out_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0].strip()

    if debug:
        print("\nRaw decoded FULL text:")
        print(full_text)

        print("\nRaw decoded NEW text only:")
        print(new_text)

    if len(new_text.strip()) > 0:
        ans = new_text
    else:
        ans = full_text.split("Label:")[-1]

    pred = clean_pred_qwen_speech_only(ans)

    if debug:
        print("Cleaned prediction:", pred)
        print("-" * 80)

    return pred

# ============================================================
# Debug run on first 5 samples
# ============================================================

print("\nRunning Qwen speech-only debug predictions on first 5 samples:")

debug_preds = []

for i in range(min(5, len(df))):
    audio_path = os.path.join(audio_dir, str(df.loc[i, audio_col]))
    true_label = str(df.loc[i, label_col])

    pred = predict_qwen_speech_only(audio_path, i=i, debug=True)
    debug_preds.append(pred)

    print("Sample:", i)
    print("True label:", true_label)
    print("Predicted label:", pred)

print("\nDebug first 5 predictions:", debug_preds)

# ============================================================
# Full dataset run
# ============================================================

print("\nNow running Qwen speech-only on full dataset...")

preds = []

for i, row in tqdm(df.iterrows(), total=len(df)):
    audio_path = os.path.join(audio_dir, str(row[audio_col]))

    pred = predict_qwen_speech_only(audio_path, i=i, debug=False)
    preds.append(pred)

df["predicted_type"] = preds

print("\nPrediction counts:")
print(df["predicted_type"].value_counts())

print("\nFirst 20 predictions:")
print(df[[audio_col, text_col, label_col, "predicted_type"]].head(20))

# ============================================================
# Metrics
# ============================================================

y_true = df[label_col].astype(str).tolist()
y_pred = df["predicted_type"].astype(str).tolist()

acc = accuracy_score(y_true, y_pred)

p, r, f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

print("Accuracy:", acc)
print("Precision:", p)
print("Recall:", r)
print("F1 Score:", f1)

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, zero_division=0))

# ============================================================
# Save predictions
# ============================================================

df.to_csv("qwen25_omni_speech_only_predictions_fixed.csv", index=False)

print("Saved predictions to qwen25_omni_speech_only_predictions_fixed.csv")

from google.colab import files
files.download("qwen25_omni_speech_only_predictions_fixed.csv")

CSV preview:
      file_name                                         transcript  \
0   0_0_d45.wav     Hello, Jane. I'm really glad to meet you here.   
1   0_0_d47.wav         I heard you and James are engaged at last.   
2  0_0_d215.wav                            Give me scotch, please.   
3   0_0_d49.wav            Hi, what brings you to my office today?   
4  0_0_d159.wav  Please help yourself at your dishes. I hope yo...   

           label  
0    declarative  
1    declarative  
2     imperative  
3  interrogative  
4     imperative  
Columns: Index(['file_name', 'transcript', 'label'], dtype='object')

Running Qwen2.5-Omni speech-only evaluation
Audio column: file_name
Label column: label
Labels: ['declarative', 'exclamatory', 'imperative', 'interrogative']
Missing audio files: 0

Checking first 3 audio files:

Sample 0
Audio path: audio_files/0_0_d45.wav
Audio exists: True
Audio size bytes: 242594
True label: declarative

Sample 1
Audio path: audio_files/0_0_d47.wav
Audio exis

100%|██████████| 404/404 [10:21<00:00,  1.54s/it]


Prediction counts:
predicted_type
declarative      154
interrogative    138
imperative        62
exclamatory       50
Name: count, dtype: int64

First 20 predictions:
        file_name                                         transcript  \
0     0_0_d45.wav     Hello, Jane. I'm really glad to meet you here.   
1     0_0_d47.wav         I heard you and James are engaged at last.   
2    0_0_d215.wav                            Give me scotch, please.   
3     0_0_d49.wav            Hi, what brings you to my office today?   
4    0_0_d159.wav  Please help yourself at your dishes. I hope yo...   
5     0_0_d50.wav                                  Hello, Jack here.   
6    0_0_d403.wav            Excuse me. I wonder if you can help me.   
7     0_0_d63.wav            Umm, Jenny, are you having a good time?   
8    0_0_d681.wav  Excuse me. I bought this shirt yesterday, but ...   
9    0_0_d684.wav     Watch out! You're too close to the fire place.   
10   0_0_d704.wav                       

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
# Qwen2.5-Omni tensor / embedding analysis
# Speech-only vs Speech+text

import os
import torch
import numpy as np
import pandas as pd

from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import files

# ------------------------------------------------------------
# Basic setup
# ------------------------------------------------------------

file_path = "test_dataset_final.csv"
audio_dir = "audio_files"

text_col = "transcript"
audio_col = "file_name"
label_col = "label"

df = pd.read_csv(file_path)
df = df.dropna(subset=[text_col, audio_col, label_col]).reset_index(drop=True)

N_ANALYSIS = 50
small_df = df.head(N_ANALYSIS).copy()

print("Running Qwen2.5-Omni tensor / embedding analysis")
print("Number of samples:", len(small_df))

if "model" not in globals() or "processor" not in globals():
    raise RuntimeError("model and processor are not loaded. Run your Qwen2.5-Omni model loading cell first.")

# ------------------------------------------------------------
# Prompts
# ------------------------------------------------------------

def make_qwen_speech_only_prompt():
    prompt = """You are a sentence type classifier.

Listen to the speech audio carefully.

Classify the spoken sentence into exactly one of these labels:

declarative: a normal statement or information-giving sentence.
interrogative: a question or sentence asking for information.
imperative: a command, request, advice, instruction, or order.
exclamatory: a sentence showing strong emotion, surprise, excitement, or emphasis.

Use only the speech audio. Do not use any transcript.

Return only one label from:
declarative, exclamatory, imperative, interrogative

Answer with only the label.

Label:"""

    return prompt

def make_qwen_speech_text_prompt(transcript):
    prompt = f"""You are a sentence type classifier.

You will receive both the speech audio and its transcript.

Classify the sentence into exactly one of these labels:

declarative: a normal statement or information-giving sentence.
interrogative: a question or sentence asking for information.
imperative: a command, request, advice, instruction, or order.
exclamatory: a sentence showing strong emotion, surprise, excitement, or emphasis.

Use both the speech audio and the transcript.

Transcript:
{transcript}

Return only one label from:
declarative, exclamatory, imperative, interrogative

Answer with only the label.

Label:"""

    return prompt

# ------------------------------------------------------------
# Build processor inputs
# Important fix: use {"type": "audio", "path": audio_path}
# ------------------------------------------------------------

def build_qwen_inputs(audio_path, prompt):
    conversation = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": "You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of perceiving auditory and visual inputs, as well as generating text and speech."
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "audio",
                    "path": audio_path
                },
                {
                    "type": "text",
                    "text": prompt
                }
            ],
        }
    ]

    try:
        inputs = processor.apply_chat_template(
            conversation,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            padding=True,
            use_audio_in_video=False
        )
    except TypeError:
        inputs = processor.apply_chat_template(
            conversation,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt"
        )

    return inputs, conversation

# ------------------------------------------------------------
# Find nested text embedding layer
# Qwen2.5-Omni top-level model.get_input_embeddings() may fail
# ------------------------------------------------------------

def find_text_embedding_layer(model):
    candidates = []

    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Embedding):
            candidates.append((name, module))

    print("\nFound embedding layers:")

    for name, module in candidates:
        print(name, "weight shape:", tuple(module.weight.shape), "device:", module.weight.device)

    if len(candidates) == 0:
        raise ValueError("No torch.nn.Embedding layer found inside the model.")

    preferred_words = [
        "embed_tokens",
        "tok_embeddings",
        "word_embeddings",
        "wte"
    ]

    for word in preferred_words:
        for name, module in candidates:
            if word in name.lower():
                print("\nUsing text embedding layer:", name)
                return module

    candidates = sorted(
        candidates,
        key=lambda x: x[1].weight.shape[0],
        reverse=True
    )

    print("\nUsing fallback text embedding layer:", candidates[0][0])
    return candidates[0][1]

text_embedding_layer = find_text_embedding_layer(model)

# ------------------------------------------------------------
# Extract mean text embedding from input_ids
# ------------------------------------------------------------

def get_text_mean_embedding(inputs):
    emb_layer = text_embedding_layer
    emb_device = emb_layer.weight.device

    input_ids = inputs["input_ids"].to(emb_device)

    with torch.no_grad():
        text_emb = emb_layer(input_ids)

    text_emb = text_emb.detach().float().cpu()

    # shape: batch, token_length, hidden_dim
    mean_emb = text_emb.mean(dim=1).squeeze(0)

    return mean_emb.numpy(), tuple(text_emb.shape)

# ------------------------------------------------------------
# Extract mean audio feature from input_features
# ------------------------------------------------------------

def get_audio_mean_feature(inputs):
    audio_keys = [
        k for k in inputs.keys()
        if "feature" in k.lower() or "audio" in k.lower()
    ]

    if "input_features" not in inputs:
        return None, None, audio_keys

    feats = inputs["input_features"].detach().float().cpu()
    raw_shape = tuple(feats.shape)

    x = feats.squeeze(0)

    if x.ndim == 2:
        # x can be frames x feature_dim or feature_dim x frames
        if x.shape[-1] in [80, 128, 256]:
            vec = x.mean(dim=0)
        elif x.shape[0] in [80, 128, 256]:
            vec = x.mean(dim=1)
        else:
            if x.shape[0] > x.shape[1]:
                vec = x.mean(dim=0)
            else:
                vec = x.mean(dim=1)

    elif x.ndim == 3:
        vec = x.reshape(-1, x.shape[-1]).mean(dim=0)

    else:
        flat = x.reshape(-1)
        vec = torch.tensor([
            flat.mean(),
            flat.std(),
            flat.min(),
            flat.max()
        ])

    return vec.numpy(), raw_shape, audio_keys

# ------------------------------------------------------------
# Run analysis
# ------------------------------------------------------------

analysis_rows = []

speech_only_text_vecs = []
speech_text_text_vecs = []
speech_only_audio_vecs = []
speech_text_audio_vecs = []

for i, row in tqdm(small_df.iterrows(), total=len(small_df)):
    audio_path = os.path.join(audio_dir, str(row[audio_col]))
    transcript = str(row[text_col])
    label = str(row[label_col])

    speech_only_prompt = make_qwen_speech_only_prompt()
    speech_text_prompt = make_qwen_speech_text_prompt(transcript)

    so_inputs, so_conversation = build_qwen_inputs(
        audio_path,
        speech_only_prompt
    )

    st_inputs, st_conversation = build_qwen_inputs(
        audio_path,
        speech_text_prompt
    )

    if i == 0:
        print("\nSpeech-only processor keys:")
        print(list(so_inputs.keys()))

        print("\nSpeech+text processor keys:")
        print(list(st_inputs.keys()))

        print("\nSpeech-only conversation content types:")
        for msg in so_conversation:
            print("Role:", msg["role"])
            for item in msg["content"]:
                print("  type:", item["type"])

    so_text_vec, so_text_shape = get_text_mean_embedding(so_inputs)
    st_text_vec, st_text_shape = get_text_mean_embedding(st_inputs)

    so_audio_vec, so_audio_shape, so_audio_keys = get_audio_mean_feature(so_inputs)
    st_audio_vec, st_audio_shape, st_audio_keys = get_audio_mean_feature(st_inputs)

    speech_only_text_vecs.append(so_text_vec)
    speech_text_text_vecs.append(st_text_vec)

    if so_audio_vec is not None:
        speech_only_audio_vecs.append(so_audio_vec)

    if st_audio_vec is not None:
        speech_text_audio_vecs.append(st_audio_vec)

    text_cos = cosine_similarity(
        so_text_vec.reshape(1, -1),
        st_text_vec.reshape(1, -1)
    )[0][0]

    if so_audio_vec is not None and st_audio_vec is not None:
        min_len = min(len(so_audio_vec), len(st_audio_vec))

        a = so_audio_vec[:min_len]
        b = st_audio_vec[:min_len]

        audio_cos = cosine_similarity(
            a.reshape(1, -1),
            b.reshape(1, -1)
        )[0][0]

        audio_abs_diff = float(np.mean(np.abs(a - b)))
    else:
        audio_cos = None
        audio_abs_diff = None

    analysis_rows.append({
        "sample_id": i,
        "file_name": row[audio_col],
        "label": label,

        "speech_only_input_ids_len": int(so_inputs["input_ids"].shape[-1]),
        "speech_text_input_ids_len": int(st_inputs["input_ids"].shape[-1]),

        "speech_only_text_embedding_shape": str(so_text_shape),
        "speech_text_text_embedding_shape": str(st_text_shape),

        "speech_only_audio_feature_shape": str(so_audio_shape),
        "speech_text_audio_feature_shape": str(st_audio_shape),

        "speech_only_audio_keys": str(so_audio_keys),
        "speech_text_audio_keys": str(st_audio_keys),

        "text_embedding_cosine_speech_only_vs_speech_text": float(text_cos),
        "audio_feature_cosine_speech_only_vs_speech_text": None if audio_cos is None else float(audio_cos),
        "audio_feature_mean_abs_diff": audio_abs_diff
    })

analysis_df = pd.DataFrame(analysis_rows)

# ------------------------------------------------------------
# Print summary
# ------------------------------------------------------------

print("\nQwen tensor / embedding analysis:")
print(analysis_df)

print("\nSummary:")

print("Average speech-only input_ids length:")
print(analysis_df["speech_only_input_ids_len"].mean())

print("\nAverage speech+text input_ids length:")
print(analysis_df["speech_text_input_ids_len"].mean())

print("\nAverage text embedding cosine speech-only vs speech+text:")
print(analysis_df["text_embedding_cosine_speech_only_vs_speech_text"].mean())

print("\nAverage audio feature cosine speech-only vs speech+text:")
print(analysis_df["audio_feature_cosine_speech_only_vs_speech_text"].mean())

print("\nAverage audio feature mean absolute difference:")
print(analysis_df["audio_feature_mean_abs_diff"].mean())

# ------------------------------------------------------------
# Save files
# ------------------------------------------------------------

analysis_df.to_csv("qwen_tensor_embedding_analysis_n50_corrected.csv", index=False)

np.save(
    "qwen_speech_only_text_mean_embeddings_n50_corrected.npy",
    np.stack(speech_only_text_vecs)
)

np.save(
    "qwen_speech_text_text_mean_embeddings_n50_corrected.npy",
    np.stack(speech_text_text_vecs)
)

if len(speech_only_audio_vecs) > 0:
    np.save(
        "qwen_speech_only_audio_mean_features_n50_corrected.npy",
        np.stack(speech_only_audio_vecs)
    )

if len(speech_text_audio_vecs) > 0:
    np.save(
        "qwen_speech_text_audio_mean_features_n50_corrected.npy",
        np.stack(speech_text_audio_vecs)
    )

print("\nSaved files:")
print("qwen_tensor_embedding_analysis_n50_corrected.csv")
print("qwen_speech_only_text_mean_embeddings_n50_corrected.npy")
print("qwen_speech_text_text_mean_embeddings_n50_corrected.npy")
print("qwen_speech_only_audio_mean_features_n50_corrected.npy")
print("qwen_speech_text_audio_mean_features_n50_corrected.npy")

files.download("qwen_tensor_embedding_analysis_n50_corrected.csv")

Running Qwen2.5-Omni tensor / embedding analysis
Number of samples: 50

Found embedding layers:
thinker.audio_tower.audio_bos_eos_token weight shape: (2, 2048) device: cuda:0
thinker.model.embed_tokens weight shape: (151936, 2048) device: cuda:0
talker.model.embed_tokens weight shape: (8448, 2048) device: cuda:0
token2wav.code2wav_dit_model.text_embed.codec_embed weight shape: (8194, 512) device: cuda:0

Using text embedding layer: thinker.model.embed_tokens


  2%|▏         | 1/50 [00:00<00:29,  1.64it/s]


Speech-only processor keys:
['input_ids', 'attention_mask', 'feature_attention_mask', 'input_features']

Speech+text processor keys:
['input_ids', 'attention_mask', 'feature_attention_mask', 'input_features']

Speech-only conversation content types:
Role: system
  type: text
Role: user
  type: audio
  type: text


100%|██████████| 50/50 [00:21<00:00,  2.34it/s]


Qwen tensor / embedding analysis:
    sample_id      file_name          label  speech_only_input_ids_len  \
0           0    0_0_d45.wav    declarative                        241   
1           1    0_0_d47.wav    declarative                        282   
2           2   0_0_d215.wav     imperative                        218   
3           3    0_0_d49.wav  interrogative                        259   
4           4   0_0_d159.wav     imperative                        291   
5           5    0_0_d50.wav    declarative                        214   
6           6   0_0_d403.wav     imperative                        236   
7           7    0_0_d63.wav  interrogative                        263   
8           8   0_0_d681.wav     imperative                        321   
9           9   0_0_d684.wav    exclamatory                        247   
10         10   0_0_d704.wav     imperative                        227   
11         11   0_0_d749.wav     imperative                        233   
12 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>